# Un clasificador de imágenes por dentro

**Charla:** IA para la detección de enfermedades foliares en arroz y musáceas mediante drones: explorando la matemática
**Proyecto:** MESCYT / FONDOCYT 2024 · UASD

Este cuaderno carga una red convolucional ya entrenada (**ResNet-50**), le muestra una foto tuya y enseña lo que ocurre por dentro: el tensor de entrada, las probabilidades de salida, los filtros que la red aprendió y la zona de la foto en la que se fijó.

**Cómo usarlo:** ejecuta las celdas en orden con el botón ▶ de cada una, o con el menú *Entorno de ejecución → Ejecutar todas*.

> La red fue entrenada con ImageNet: reconoce 1,000 categorías cotidianas (animales, objetos, alimentos) y los nombres están en inglés. **No** conoce enfermedades del arroz; para eso hay que entrenarla con imágenes del cultivo, que es lo que hace el proyecto.

## 1. Cargar la red

La primera vez descarga unos 100 MB con los pesos: los millones de números que la red aprendió durante su entrenamiento.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision.models import ResNet50_Weights, resnet50

pesos = ResNet50_Weights.IMAGENET1K_V1
modelo = resnet50(weights=pesos).eval()
clases = pesos.meta["categories"]
preparar = pesos.transforms()

total = sum(p.numel() for p in modelo.parameters())
print(f"Red cargada: ResNet-50 con {total:,} parámetros y {len(clases)} clases.")

## 2. Subir una foto

Elige cualquier imagen de tu equipo. Si cancelas, o si no estás en Colab, se usa una foto de ejemplo.

In [ ]:
def foto_de_ejemplo() -> Image.Image:
    from skimage import data
    return Image.fromarray(data.chelsea())

try:
    from google.colab import files
    subidos = files.upload()
    nombre = next(iter(subidos))
    imagen = Image.open(nombre).convert("RGB")
    print("Foto cargada:", nombre)
except Exception:
    imagen = foto_de_ejemplo()
    print("Usando la foto de ejemplo.")

plt.figure(figsize=(5, 5))
plt.imshow(imagen)
plt.axis("off")
plt.title(f"Imagen original: {imagen.width} × {imagen.height} píxeles")
plt.show()

## 3. La imagen como tensor

Para la red, la foto es un bloque de números con forma **canales × alto × ancho**. Aquí son 3 canales (rojo, verde, azul) de 224 × 224. En el proyecto son 4 canales: verde, rojo, borde rojo e infrarrojo cercano.

In [ ]:
x = preparar(imagen)          # recorta a 224 x 224 y normaliza
print("Forma del tensor:", tuple(x.shape))
print("\nEsquina superior izquierda del canal rojo (5 x 5 números):")
print(np.round(x[0, :5, :5].numpy(), 2))

fig, ejes = plt.subplots(1, 3, figsize=(10, 3.6))
for eje, canal, nombre, mapa in zip(ejes, x, ["Rojo", "Verde", "Azul"], ["Reds", "Greens", "Blues"]):
    eje.imshow(canal, cmap=mapa)
    eje.set_title(f"Canal {nombre}")
    eje.axis("off")
plt.suptitle("Tres matrices apiladas = un tensor")
plt.tight_layout()
plt.show()

## 4. La predicción: puntajes → softmax → probabilidades

La red entrega un **puntaje** por cada una de las 1,000 clases. *Softmax* los convierte en probabilidades que suman 1.

In [ ]:
with torch.no_grad():
    puntajes = modelo(x.unsqueeze(0))[0]
probabilidades = F.softmax(puntajes, dim=0)

mejores = torch.topk(probabilidades, 5)
print(f"Suma de las 1,000 probabilidades: {probabilidades.sum().item():.4f}\n")
for p, i in zip(mejores.values, mejores.indices):
    print(f"{clases[i]:<28} puntaje = {puntajes[i].item():6.2f}   probabilidad = {p.item():.3f}")

nombres = [clases[i] for i in mejores.indices][::-1]
valores = mejores.values.numpy()[::-1]
plt.figure(figsize=(7, 3.2))
plt.barh(nombres, valores, color="#1f6b3a")
plt.xlim(0, 1)
plt.xlabel("probabilidad")
plt.title("Las 5 clases más probables")
for y, v in enumerate(valores):
    plt.text(v + 0.01, y, f"{v:.2f}", va="center")
plt.tight_layout()
plt.show()

## 5. Los filtros que la red aprendió

Estos son los 64 filtros de la **primera capa convolucional** (cada uno de 7 × 7 píxeles). Nadie los programó: salieron del entrenamiento. Fíjate en que son detectores de bordes en distintas direcciones y de contrastes de color.

In [ ]:
filtros = modelo.conv1.weight.detach().clone()          # 64 x 3 x 7 x 7
filtros = (filtros - filtros.min()) / (filtros.max() - filtros.min())

fig, ejes = plt.subplots(8, 8, figsize=(7, 7))
for eje, f in zip(ejes.flat, filtros):
    eje.imshow(f.permute(1, 2, 0), interpolation="nearest")
    eje.axis("off")
plt.suptitle("64 filtros de la primera capa (aprendidos, no diseñados)")
plt.tight_layout()
plt.show()

## 6. Qué "ve" cada capa

Pasamos la foto por la red y nos detenemos en tres puntos. Al avanzar, los mapas son **más pequeños** (pooling) y responden a patrones **más abstractos**: primero bordes, luego texturas, al final partes del objeto.

In [ ]:
def mostrar_mapas(mapas: torch.Tensor, titulo: str, cuantos: int = 8) -> None:
    # Los canales con mayor respuesta media son los más informativos para mostrar.
    orden = mapas.mean(dim=(1, 2)).argsort(descending=True)[:cuantos]
    fig, ejes = plt.subplots(1, cuantos, figsize=(1.6 * cuantos, 2))
    for eje, i in zip(ejes, orden):
        eje.imshow(mapas[i], cmap="viridis")
        eje.axis("off")
    plt.suptitle(f"{titulo}  ·  {mapas.shape[0]} mapas de {mapas.shape[1]} × {mapas.shape[2]}")
    plt.tight_layout()
    plt.show()

with torch.no_grad():
    a = modelo.relu(modelo.bn1(modelo.conv1(x.unsqueeze(0))))     # conv + ReLU
    mostrar_mapas(a[0], "Primera capa: bordes")
    a = modelo.layer1(modelo.maxpool(a))                            # + pooling
    a = modelo.layer2(a)
    mostrar_mapas(a[0], "Capa intermedia: texturas")
    a = modelo.layer4(modelo.layer3(a))
    mostrar_mapas(a[0], "Última capa: partes del objeto")

## 7. ¿En qué parte de la foto se fijó?

Usamos el **gradiente** (la misma derivada del descenso del gradiente) para preguntar: ¿qué zonas de la imagen hicieron subir el puntaje de la clase ganadora? La técnica se llama Grad-CAM.

In [ ]:
guardado = {}
gancho = modelo.layer4.register_forward_hook(lambda capa, entrada, salida: guardado.update(mapas=salida))

salida = modelo(x.unsqueeze(0))
ganadora = salida[0].argmax().item()
gradiente = torch.autograd.grad(salida[0, ganadora], guardado["mapas"])[0]
gancho.remove()

importancia = gradiente.mean(dim=(2, 3), keepdim=True)              # un peso por mapa
calor = F.relu((importancia * guardado["mapas"]).sum(dim=1, keepdim=True))
calor = F.interpolate(calor, size=x.shape[1:], mode="bilinear", align_corners=False)[0, 0]
calor = (calor / (calor.max() + 1e-8)).detach().numpy()

# Deshacer la normalización para mostrar la foto tal como la ve la red.
media = torch.tensor(preparar.mean).view(3, 1, 1)
desviacion = torch.tensor(preparar.std).view(3, 1, 1)
visible = (x * desviacion + media).clamp(0, 1).permute(1, 2, 0).numpy()

fig, ejes = plt.subplots(1, 2, figsize=(9, 4.6))
ejes[0].imshow(visible)
ejes[0].set_title("Lo que recibe la red (224 × 224)")
ejes[1].imshow(visible)
ejes[1].imshow(calor, cmap="jet", alpha=0.45)
ejes[1].set_title(f"Zona decisiva para: {clases[ganadora]}")
for eje in ejes:
    eje.axis("off")
plt.tight_layout()
plt.show()

## Para llevar

- La foto entró como un **tensor** de 3 × 224 × 224 números.
- Pasó por decenas de **convoluciones** con filtros aprendidos, cada una seguida de **ReLU**.
- Salieron 1,000 puntajes que **softmax** convirtió en probabilidades.
- El **gradiente** nos dijo qué píxeles importaron.

En el proyecto usamos las mismas piezas con dos cambios: la entrada tiene **4 bandas** del dron multiespectral y la salida es un **mapa** con una probabilidad por píxel (arquitectura U-Net).

Prueba con otra foto: vuelve a ejecutar desde la celda 2.